# EDA — Atlas do Desenvolvimento Humano e FUNDEB (NSE)

**Escopo deste notebook:** análise exploratória das colunas de contexto que
eu contribuí para `base_analitica` — `ctx_atlas_*` (35 indicadores do Atlas
do Desenvolvimento Humano) e `ctx_fundeb_*` (Nível Socioeconômico dos entes
federados). Outras fontes (INEP, IBGE, Censo Escolar) são analisadas nos
EDAs dos respectivos responsáveis.

**Fontes:**
- Atlas do Desenvolvimento Humano — espelho público (`github.com/mauriciocramos/IDHM`),
  réplica do arquivo oficial PNUD/IPEA/Fundação João Pinheiro. Ano-base 2010
  (Censo Demográfico), fixo para 2023-2025.
- FUNDEB — Nível Socioeconômico (NSE) por ente federado, publicado pelo INEP.
  Cobre 2024 e 2025; **2023 fica nulo, sem proxy** (decisão validada
  empiricamente: o NSE varia significativamente entre edições, não é um
  atributo estrutural estável — ver `src/preprocessing/join.py`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')


## 1. Carregando a base

In [ ]:
# base_analitica é particionada por ano em data/processed/
dest = Path('../data/processed/base_analitica')

partes = sorted(dest.glob('ano=*'))
dfs = []
for p in partes:
    ano = int(p.name.split('=')[1])
    sub = pd.read_parquet(p)
    sub['ano'] = ano
    dfs.append(sub)

df = pd.concat(dfs, ignore_index=True)
print(f'{len(df):,} linhas x {len(df.columns)} colunas')
print('Anos:', sorted(df['ano'].unique()))


In [ ]:
ATLAS_COLS = [c for c in df.columns if c.startswith('ctx_atlas_')]
FUNDEB_COLS = [c for c in df.columns if c.startswith('ctx_fundeb_')]

print(f'{len(ATLAS_COLS)} colunas do Atlas')
print(f'{len(FUNDEB_COLS)} colunas do FUNDEB')


## 2. Valores faltantes

O FUNDEB só publica NSE a partir de 2024 — a expectativa é **2023 = 100% nulo**
nessas colunas, por decisão de design (não é falha de match). O Atlas está
fixo em 2010 e é replicado para todos os anos, então a expectativa é
**0% nulo** em qualquer ano.


In [ ]:
nulos_por_ano = df.groupby('ano')[ATLAS_COLS + FUNDEB_COLS].apply(
    lambda g: g.isna().mean() * 100
)
nulos_por_ano[['ctx_atlas_idhm', 'ctx_fundeb_nse_municipio', 'ctx_fundeb_nse_uf']].round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
nulos_por_ano[['ctx_atlas_idhm', 'ctx_fundeb_nse_municipio']].plot(kind='bar', ax=ax)
ax.set_ylabel('% nulo')
ax.set_title('Cobertura por ano — Atlas (esperado: sempre 0%) vs FUNDEB (esperado: 100% em 2023)')
ax.legend(['Atlas (idhm)', 'FUNDEB (nse_municipio)'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 3. Distribuições

Olhando a forma de alguns indicadores-chave — síntese (IDHM), renda,
pobreza, saúde infantil, frequência escolar e NSE.


In [ ]:
indicadores_chave = [
    'ctx_atlas_idhm',
    'ctx_atlas_renda_per_capita',
    'ctx_atlas_percentual_pobres',
    'ctx_atlas_mortalidade_ate_5_anos',
    'ctx_atlas_taxa_fora_escola_6a14',
    'ctx_fundeb_nse_municipio',
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, indicadores_chave):
    df[col].dropna().hist(bins=30, ax=ax, edgecolor='white')
    ax.set_title(col.replace('ctx_atlas_', '').replace('ctx_fundeb_', ''))
plt.tight_layout()
plt.show()


## 4. Correlação entre os indicadores do Atlas

Vale destacar: os subíndices do IDHM (educação, renda, longevidade) tendem
a ser fortemente correlacionados entre si por construção — o IDHM é a média
geométrica dos três. Também esperamos alta correlação entre os indicadores
de pobreza (`percentual_pobres`, `percentual_extremamente_pobres`,
`percentual_vulneraveis_pobreza`), já que são aninhados por definição
(extremamente pobres ⊆ pobres ⊆ vulneráveis).


In [ ]:
corr = df[ATLAS_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, square=True,
            xticklabels=True, yticklabels=True)
ax.set_title('Correlação entre indicadores do Atlas')
plt.tight_layout()
plt.show()


## 5. Relação com o alvo (`label_alfabetizado`)

Comparamos a média de cada indicador entre alunos alfabetizados (1) e não
alfabetizados (0), ordenado pela maior diferença absoluta — um primeiro
sinal (não causal) de quais variáveis têm mais associação com o resultado.

⚠️ **Cuidado ao interpretar**: essa análise é sobre o *contexto municipal*
do aluno, que é compartilhado por todos os alunos do mesmo município — não é
uma característica individual. Diferenças aqui refletem padrões
socioeconômicos regionais associados à alfabetização, não necessariamente
causas diretas.


In [ ]:
medias_por_label = df.groupby('label_alfabetizado')[ATLAS_COLS + FUNDEB_COLS].mean().T
medias_por_label.columns = ['nao_alfabetizado', 'alfabetizado']
medias_por_label['diferenca'] = medias_por_label['alfabetizado'] - medias_por_label['nao_alfabetizado']

ranking = medias_por_label.reindex(
    medias_por_label['diferenca'].abs().sort_values(ascending=False).index
)
ranking.head(15).round(3)


In [ ]:
top10 = ranking.head(10)

fig, ax = plt.subplots(figsize=(9, 6))
cores = ['#2a9d8f' if v > 0 else '#e76f51' for v in top10['diferenca']]
ax.barh(top10.index.str.replace('ctx_atlas_', '').str.replace('ctx_fundeb_', ''),
        top10['diferenca'], color=cores)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Diferença de média (alfabetizado − não alfabetizado)')
ax.set_title('Top 10 indicadores do Atlas/FUNDEB por diferença absoluta vs. alvo')
plt.tight_layout()
plt.show()


In [ ]:
indicadores_boxplot = ['ctx_atlas_idhm', 'ctx_atlas_percentual_pobres', 'ctx_fundeb_nse_municipio']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, indicadores_boxplot):
    df.boxplot(column=col, by='label_alfabetizado', ax=ax)
    ax.set_title(col.replace('ctx_atlas_', '').replace('ctx_fundeb_', ''))
    ax.set_xlabel('label_alfabetizado')
plt.suptitle('')
plt.tight_layout()
plt.show()


## 6. Padrões geográficos

Média do IDHM por região — um primeiro olhar descritivo antes de uma
clusterização formal (que pode ser um próximo passo, se o time achar
relevante para a pergunta "quais regiões têm padrões semelhantes?").


In [ ]:
# Código IBGE de UF -> (sigla, região). Tabela estática, estável.
UF_INFO = {
    11: ('RO', 'Norte'), 12: ('AC', 'Norte'), 13: ('AM', 'Norte'), 14: ('RR', 'Norte'),
    15: ('PA', 'Norte'), 16: ('AP', 'Norte'), 17: ('TO', 'Norte'),
    21: ('MA', 'Nordeste'), 22: ('PI', 'Nordeste'), 23: ('CE', 'Nordeste'),
    24: ('RN', 'Nordeste'), 25: ('PB', 'Nordeste'), 26: ('PE', 'Nordeste'),
    27: ('AL', 'Nordeste'), 28: ('SE', 'Nordeste'), 29: ('BA', 'Nordeste'),
    31: ('MG', 'Sudeste'), 32: ('ES', 'Sudeste'), 33: ('RJ', 'Sudeste'), 35: ('SP', 'Sudeste'),
    41: ('PR', 'Sul'), 42: ('SC', 'Sul'), 43: ('RS', 'Sul'),
    50: ('MS', 'Centro-Oeste'), 51: ('MT', 'Centro-Oeste'),
    52: ('GO', 'Centro-Oeste'), 53: ('DF', 'Centro-Oeste'),
}

df['regiao'] = df['id_uf'].map(lambda x: UF_INFO.get(x, ('??', '??'))[1])

by_region = df.groupby('regiao')[['ctx_atlas_idhm', 'ctx_fundeb_nse_municipio']].mean().sort_values('ctx_atlas_idhm')
by_region


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
by_region['ctx_atlas_idhm'].plot(kind='barh', ax=ax, color='#264653')
ax.set_xlabel('IDHM médio')
ax.set_title('IDHM médio por região (contexto dos alunos)')
plt.tight_layout()
plt.show()


## 7. Resumo dos achados

*(preencher após rodar com dado real — os números abaixo são só a estrutura
de como cada seção deve ser reportada)*

- **Cobertura**: Atlas 100% preenchido em todos os anos; FUNDEB nulo em
  2023 (esperado, decisão de design documentada em `join.py`).
- **Indicadores mais associados ao alvo**: ver ranking da seção 5 —
  atualizar com os nomes/valores reais após rodar.
- **Padrão geográfico**: ver seção 6 — atualizar com a leitura real.
- **Limitações**: o Atlas é de 2010 (13-15 anos antes dos dados de aluno);
  o NSE do FUNDEB não tem proxy para 2023, reduzindo a cobertura de
  features nesse ano especificamente.

## Próximos passos sugeridos

- Repetir a análise da seção 5 controlando por UF/região, para separar
  efeito de nível de desenvolvimento vs. efeito puramente geográfico.
- Cruzar com o EDA de outras fontes (INEP, IBGE, Censo Escolar) para ver
  se os indicadores mais associados ao alvo se repetem ou se
  complementam entre fontes.
- Avaliar se a ausência de NSE em 2023 deve virar uma feature binária
  explícita ("tem_nse") no pipeline de modelagem, em vez de só um NaN.
